In [0]:
/*
  Script: Profile Silver
  Description:
    This script
*/

/*
================================================================
Table: crm_cust_info
================================================================
*/

/*
  Data Overview
*/
SELECT *
FROM bronze.crm_cust_info
LIMIT 50
;


/*
  Uniqueness Constraint
  Result: Success. There are no duplicate rows (19494).
*/
SELECT
  COUNT(DISTINCT STRUCT(*)) AS total_unique_rows,
  COUNT(*) AS total_rows
FROM bronze.crm_cust_info
;


/*
  Uniqueness Constraint
  Result: Fail. There are duplicate cst_ids.
*/
SELECT
  COUNT(DISTINCT cst_id) AS total_cst_id,
  COUNT(*) AS total_rows
FROM bronze.crm_cust_info
;


/*
  Uniqueness Constraint
  Result: Fail. There are duplicate cst_keys.
*/
SELECT
  COUNT(DISTINCT cst_key) AS total_cst_key,
  COUNT(*) AS total_rows
FROM bronze.crm_cust_info
;


/*
  Conclusion: cst_ids are more unique than cst_keys.
*/

/*
  Deep-dive: What is the meaning of _id and _key? Why are they duplicated?
  Result: Highly likely that it's a data error.
    There is only one cst_id with multiple cst_key values.
    This cst_id is null.
    The associated cst_key value does not conform to the typical format of other cst_keys.
    Other values in the column is also completely null.
  Conclusion: Can delete the rows with this null cst_id and non-standar cst_keys (4 rows).
*/
SELECT
  cst_id,
  collect_set(cst_key) AS list_key,
  size(collect_set(cst_key)) AS count_key
FROM bronze.crm_cust_info
GROUP BY cst_id
HAVING count_key > 1
ORDER BY count_key DESC
;

SELECT
  *
FROM bronze.crm_cust_info
WHERE cst_id IS NULL;

/*
  Uniqueness Constraint
  Result: Fail. There are duplicate combinations.
*/
SELECT
  COUNT(DISTINCT STRUCT(cst_key, cst_id)) AS total_cst_com,
  COUNT(*) AS total_rows
FROM bronze.crm_cust_info
;


-- There are no duplicate rows, but the keys and ids are duplicated.
/*
  Hypothesis Check: There are different versions of the keys and ids by date
  Result: True. Some combinations have different versions by date.
*/
SELECT
  STRUCT(cst_key, cst_id) AS cst_com,
  COUNT(DISTINCT cst_create_date) AS count_date
FROM bronze.crm_cust_info
GROUP BY cst_com
ORDER BY count_date DESC
;


/*
  Hypothesis Follow-up: The info of each key, id combination differ by date.
  Result: True. But the difference seem to be that null values get updated to non-null values.
*/
WITH t AS
(
  SELECT 
    STRUCT(cst_key, cst_id) AS cst_com,
    COUNT(DISTINCT cst_create_date) AS count_date
  FROM bronze.crm_cust_info
  GROUP BY cst_com
  HAVING count_date > 1
)
SELECT
  *
FROM
  t
  INNER JOIN
  (
    SELECT
      STRUCT(cst_key, cst_id) AS cst_com,
      *
    FROM
      bronze.crm_cust_info
  )
  USING (cst_com)
ORDER BY
  cst_com,
  cst_create_date
DESC
;

/*
-- Fun fact: I was suggested this query by Genie, but it's actually slower than the above query that I wrote. (400 vs. 600 ms)
SELECT c.*, d.date_count
FROM bronze.crm_cust_info c
INNER JOIN (
    SELECT cst_key, cst_id, COUNT(DISTINCT cst_create_date) AS date_count
    FROM bronze.crm_cust_info
    GROUP BY cst_key, cst_id
    HAVING COUNT(DISTINCT cst_create_date) > 1
) d ON c.cst_key = d.cst_key AND c.cst_id = d.cst_id;
*/

/*
  Hypothesis Follow-up: The only difference between the versions is that null values get updated to non-null values.
  Result: True. There are no new, rewritten values.
*/
SELECT
  cst_key,
  cst_id,
  COUNT(DISTINCT cst_create_date) AS count_date,
  size(collect_set(cst_firstname)) AS firstname_count,
  size(collect_set(cst_lastname)) AS lastname_count,
  size(collect_set(cst_marital_status)) AS marital_status_count,
  size(collect_set(cst_gndr)) AS gndr_count
FROM bronze.crm_cust_info
GROUP BY cst_key, cst_id
HAVING count_date > 1;

/*
  Uniqueness Contraint
  Conclusion: Only use the newest version of the data to deduplicate in silver layer.
*/

/*
  Normalization
*/

-- Find rows with unwanted spaces in string columns
SELECT
  *,
  TRIM(cst_firstname) AS cst_firstname,
  TRIM(cst_lastname) AS cst_lastname,
  TRIM(cst_marital_status) AS cst_marital_status,
  TRIM(cst_gndr) AS cst_gndr
FROM bronze.crm_cust_info
;

-- Find values in cst_gndr
SELECT
  DISTINCT cst_gndr
FROM bronze.crm_cust_info
;

/*
  Data Validity
  Result: Succcess. All values match.
*/
-- Check values in cst_key that do not match with corresponding cst_id
SELECT
*
FROM bronze.crm_cust_info
WHERE cst_id != SUBSTRING(cst_key, 5)
;

-- Check values in cst_key that do not start with the correct prefix
SELECT
*
FROM bronze.crm_cust_info
WHERE cst_key NOT LIKE 'AW000%'
;

/*
================================================================
Table: crm_prd_info
================================================================
*/

/*
  Data Overview
*/
SELECT *
FROM bronze.crm_prd_info
LIMIT 50
;

/*
  Uniqueness Constraint
  Result: Success. There are no duplicate rows.
*/
SELECT
  COUNT(DISTINCT STRUCT(*)) AS total_unique_rows,
  COUNT(*) AS total_rows
FROM bronze.crm_prd_info
;


/*
  Uniqueness Constraint
  Result: Success. There are no duplicate prd_ids.
*/
SELECT
  COUNT(DISTINCT prd_id) AS total_prd_id,
  COUNT(*) AS total_rows
FROM bronze.crm_prd_info
;


/*
  Uniqueness Constraint
  Result: Fail. There are duplicate prd_keys.
*/
SELECT
  COUNT(DISTINCT prd_key) AS total_prd_key,
  COUNT(*) AS total_rows
FROM bronze.crm_prd_info
;


/*
  Uniqueness Constraint
  Result: Success. There are no duplicate combinations.
*/
SELECT
  COUNT(DISTINCT STRUCT(prd_id, prd_key)) AS total_prd_com,
  COUNT(*) AS total_rows
FROM bronze.crm_prd_info
;

/*
  Conclusion: prd_keys are more unique than prd_ids.
*/

/*
  Hypothesis Check: Rows with prd_key and prd_id values that are not injective (each value correspond to one unique value) have errors and can be removed.
  Result: False.
    The prd_key and prd_id conforms to the typical format.
    Detail values are mostly not null.
    The newest version of the prd_start_dt seem to mean the newest version. In such instances, the prd_end_dt is null.
  Conclusion:
    Keep all rows.
    This table implements SCD Type 2, with prd_id representing the unique item instance in time.
*/

-- List items with duplicated prd_ids, the list of duplicate values, and the number of duplicates.
SELECT
  prd_key,
  collect_set(prd_id) AS list_id,
  size(collect_set(prd_id)) AS count_id
FROM bronze.crm_prd_info
GROUP BY prd_key
HAVING count_id > 1
ORDER BY count_id DESC
;

-- List all details on items with duplicated prd_ids.
WITH t AS (
  SELECT
    prd_key,
    collect_set(prd_id) AS list_id,
    size(collect_set(prd_id)) AS count_id
  FROM bronze.crm_prd_info
  GROUP BY prd_key
  HAVING count_id > 1
  ORDER BY count_id DESC
)
SELECT
  *
FROM bronze.crm_prd_info
  INNER JOIN t
  USING (prd_key)
ORDER BY prd_key
;

/*
  Uniqueness Constraint
  Result: Success. There are no duplicate combinations.
*/
SELECT
  COUNT(DISTINCT STRUCT(prd_key, prd_id)) AS total_prd_com,
  COUNT(*) AS total_rows
FROM bronze.crm_prd_info
;

/*
  Data Validity
  No negative costs, only some null values.
*/
SELECT
  prd_cost
FROM bronze.crm_prd_info
WHERE prd_cost < 0 or prd_cost IS NULL
;

/*
  Data Validity: prd_start_dt and prd_end_dt
  Has some invalid date ranges (start date > end date)
*/

-- Find rows where prd_start_dt is after prd_end_dt
SELECT
  *
FROM bronze.crm_prd_info
WHERE prd_start_dt > prd_end_dt
;

-- Find rows where prd_start_dt is null
SELECT
  *
FROM bronze.crm_prd_info
WHERE prd_start_dt IS NULL
;


/*
================================================================
Table: crm_sales_details
================================================================
*/

/*
  Data Overview
*/
SELECT *
FROM bronze.crm_sales_details
LIMIT 50
;

/*
  Uniqueness Constraint
  Result: Success. There are no duplicate rows.
*/
SELECT
  COUNT(DISTINCT STRUCT(*)) AS total_unique_rows,
  COUNT(*) AS total_rows
FROM bronze.crm_sales_details
;


/*
  Uniqueness Constraint
  Result: Failure. There are many duplicate sls_ord_num.
  Problem: This table does not have a truly unique primary key.
*/
SELECT
  COUNT(DISTINCT sls_ord_num) AS total_ord_id,
  COUNT(*) AS total_rows
FROM bronze.crm_sales_details
;

/*
  Uniqueness Constraint
  Result: Success. There are no duplicate combinations.

*/
SELECT
  COUNT(DISTINCT STRUCT(sls_ord_num, sls_cust_id, sls_prd_key)) AS total_ord_com,
  COUNT(*) AS total_rows
FROM bronze.crm_sales_details
;

/*
  Hypothesis Check: Each order corresponds to one cust_id only, but one cust_id can have many orders (1-M relationship).
  Result: True.
*/
-- Find the number of sls_cust_id values associated with a sls_ord_num
SELECT
  sls_ord_num,
  size(collect_set(sls_cust_id)) AS count_cust
FROM bronze.crm_sales_details
GROUP BY sls_ord_num
HAVING count_cust > 1
ORDER BY count_cust DESC
;

-- Find the number of sls_ord_num values associated with a sls_cust_id
SELECT
  sls_cust_id,
  size(collect_set(sls_ord_num)) AS count_ord
FROM bronze.crm_sales_details
GROUP BY sls_cust_id
HAVING count_ord > 1
ORDER BY count_ord DESC
;


/*
  Hypothesis Check: Each order can correspond to many products, and each product can be associated with many orders.
  Result: True
*/
-- Find the number of count_prd values associated with a sls_ord_num
SELECT
  sls_ord_num,
  size(collect_set(sls_prd_key)) AS count_prd
FROM bronze.crm_sales_details
GROUP BY sls_ord_num
HAVING count_prd > 1
ORDER BY count_prd DESC
;

-- Find the number of sls_ord_num values associated with a sls_prd_key
SELECT
  sls_prd_key,
  size(collect_set(sls_ord_num)) AS count_ord
FROM bronze.crm_sales_details
GROUP BY sls_prd_key
HAVING count_ord > 1
ORDER BY count_ord DESC
;

/*
  Conclusion: The crm_sales_details is a flattened fact table with grain: one row per item of an order instance by a user.
*/

/*
================================================================
Table: erp_cust_az12
================================================================
*/

/*
  Data Overview
*/
SELECT *
FROM bronze.erp_cust_az12
LIMIT 50
;

/*
  Uniqueness Constraint
  Result: Success. There are no duplicate rows. (18484)
*/
SELECT
  COUNT(DISTINCT STRUCT(*)) AS total_unique_rows,
  COUNT(*) AS total_rows
FROM bronze.erp_cust_az12
;

/*
  Uniqueness Constraint
  Result: Success. There are no duplicate ids.
*/
SELECT
  COUNT(DISTINCT cid) AS total_cid,
  COUNT(*) AS total_rows
FROM bronze.erp_cust_az12
;

/*
  Normalization: ID format
  Hypothesis Check: All IDs in erp_cust_az12 have the same prefixes and corresponds to IDs in crm_cust_info
*/

-- Check the number of cids by prefixes (3 letters)
SELECT
  LEFT(cid, 3) AS cid_prefix,
  COUNT(DISTINCT cid) AS total_cid
FROM bronze.erp_cust_az12
GROUP BY cid_prefix
ORDER BY total_cid DESC
;

-- Check the number of cids by prefixes (8 letters)
SELECT
  LEFT(cid, 8) AS cid_prefix,
  COUNT(DISTINCT cid) AS total_cid
FROM bronze.erp_cust_az12
GROUP BY cid_prefix
ORDER BY total_cid DESC
;

-- Check the number of cids that naturally match cst_keys (7448)
SELECT
  COUNT(STRUCT(*))
FROM
(
  SELECT
    e.cid,
    c.cst_key
  FROM bronze.erp_cust_az12 e
  INNER JOIN bronze.crm_cust_info c
    ON e.cid = c.cst_key
)
;

-- Check the number of cids that match cst_keys *after pruning prefixes on erp_cust_az12* (11042)
SELECT
  COUNT(STRUCT(*))
FROM
(
  SELECT
    e.cid,
    SUBSTRING(e.cid, 4) AS norm_cid,
    c.cst_key
  FROM bronze.erp_cust_az12 e
  INNER JOIN bronze.crm_cust_info c
    ON SUBSTRING(e.cid, 4) = c.cst_key
)
;

-- Check the number of cids that match cst_keys *after pruning prefixes on both tables* (11042)
SELECT
  COUNT(STRUCT(*))
FROM
(
  SELECT
    e.cid,
    SUBSTRING(e.cid, 9) AS norm_cid,
    c.cst_id
  FROM bronze.erp_cust_az12 e
  INNER JOIN bronze.crm_cust_info c
    ON SUBSTRING(e.cid, 9) = c.cst_id
)
;

/*
  Conclusion:
    The behavior on crm_cust_info is pretty consistent (cst_id and cst_key map to each other).
    However, the format of cid in erp_cust_az12 is not. It does not use a consistent letters of prefixes.
*/

-- 7448 + 11042 does not equal the original number of rows.
/*
  Hypothesis Check: There are duplicate values in cid.
  Result: True.
*/

-- Find the overlap cids of the above 2 subqueries and their total number of instances in the table.
SELECT
  e.cid,
  SUBSTRING(e.cid, 4) AS norm_cid,
  collect_set(
    CASE WHEN e.cid = c.cst_key THEN c.cst_key END
  ) AS direct_matches,
  collect_set(
    CASE WHEN SUBSTRING(e.cid, 4) = c.cst_key THEN c.cst_key END
  ) AS substring_matches,
  COUNT(*) AS cnt
FROM bronze.erp_cust_az12 e
JOIN bronze.crm_cust_info c
  ON e.cid = c.cst_key
  OR SUBSTRING(e.cid, 4) = c.cst_key
GROUP BY e.cid, norm_cid
HAVING COUNT(*) > 1;

-- Check rows in cst_id that is missing cid values
SELECT
  CASE
    WHEN e.cid LIKE 'NAS%' THEN SUBSTRING(e.cid, 4)
    ELSE e.cid
  END AS cid,
  c.cst_id
FROM bronze.erp_cust_az12 e
LEFT JOIN bronze.crm_cust_info c
  ON SUBSTRING(e.cid, 9) = c.cst_id
  WHERE c.cst_id IS NULL
;

-- Check rows in cid that is missing cst_id values
SELECT
  CASE
    WHEN e.cid LIKE 'NAS%' THEN SUBSTRING(e.cid, 4)
    ELSE e.cid
  END AS cid,
  c.cst_id
FROM bronze.erp_cust_az12 e
RIGHT JOIN bronze.crm_cust_info c
  ON SUBSTRING(e.cid, 9) = c.cst_id
  WHERE e.cid IS NULL
;

/*
================================================================
Table: erp_loc_a101
================================================================
*/

/*
  Data Overview
*/
SELECT *
FROM bronze.erp_loc_a101
LIMIT 50
;

/*
  Uniqueness Constraint
  Result: Success. There are no duplicate rows (18484).
*/
SELECT
  COUNT(DISTINCT STRUCT(*)) AS total_unique_rows,
  COUNT(*) AS total_rows
FROM bronze.erp_loc_a101
;

/*
  Uniqueness Constraint
  Result: Success. There are no duplicate ids.
*/
SELECT
  COUNT(DISTINCT cid) AS total_cid,
  COUNT(*) AS total_rows
FROM bronze.erp_loc_a101
;

/*
  Relationship
  Hypothesis Check: This tables cid values entirely match with values in erp_cust_az12.
  Result: False.
    The cid in erp_loc_a101 needs to be modified (remove - symbols). After modifying it:
      A portion matches with cid in erp_cust_az12 *with prefixes* (the original format).
      A portion matches with cid in erp_cust_az12 *without prefixes* (the modified format).
*/

-- With prefix: Find the number of cleaned cid values with matches in erp_cust_az12 (7442).
SELECT
  COUNT(*)
FROM
(
  SELECT REPLACE(cid, '-', '') AS cid
  FROM bronze.erp_loc_a101
)
INNER JOIN bronze.erp_cust_az12
USING (cid)
;

-- Without prefix: Find the number of cleaned cid values with matches in erp_cust_az12 (11042).
SELECT
  COUNT(*)
FROM
(
  SELECT REPLACE(cid, '-', '') AS cid
  FROM bronze.erp_loc_a101
)
INNER JOIN
(
  SELECT SUBSTRING(cid, 4) AS cid
  FROM bronze.erp_cust_az12
)
USING (cid)
;

/*
  Relationship
  Hypothesis Check: This tables cid values entirely match with values in cst_key of crm_cust_info.
  Result: True.
    There are only a few exceptions where they do not match.
    In such cases, it is because crm_cust_info has erroneous values that must be removed beforehand.
*/

-- Find the number of cleaned cid values with matches in in cst_key of crm_cust_info (19490).
SELECT
  COUNT(*)
FROM
(
  SELECT REPLACE(cid, '-', '') AS cid_norm
  FROM bronze.erp_loc_a101
) e
INNER JOIN bronze.crm_cust_info c
ON e.cid_norm = c.cst_key
;

-- Find unmatched values.
SELECT
  *
FROM
(
  SELECT
    *,
    REPLACE(cid, '-', '') AS cid_norm
  FROM bronze.erp_loc_a101
) e
RIGHT JOIN bronze.crm_cust_info c
ON e.cid_norm = c.cst_key
WHERE e.cid_norm IS NULL
;

/*
  Data Validity: cntry
*/
SELECT DISTINCT cntry
FROM bronze.erp_loc_a101
ORDER BY cntry
;

/*
================================================================
Table: erp_px_cat_g1v2
================================================================
*/

/*
  Data Overview
*/
SELECT *
FROM bronze.erp_px_cat_g1v2
LIMIT 50
;

/*
  Uniqueness Constraint
  Result: Success. There are no duplicate rows.
*/
SELECT
  COUNT(DISTINCT STRUCT(*)) AS total_unique_rows,
  COUNT(*) AS total_rows
FROM bronze.erp_px_cat_g1v2
;

/*
  Uniqueness Constraint
  Result: Success. There are no duplicate ids.
*/
SELECT
  COUNT(DISTINCT id) AS total_cid,
  COUNT(*) AS total_rows
FROM bronze.erp_px_cat_g1v2
;

/*
  Relationship
  Hypothesis Check: This tables id values match with prefixes of prd_key in crm_prd_info.
  Result:
    There are 295 unique prd_key and 397 rows in crm_prd_info.
    There are 390 matches between prd_key and id in erp_px_cat_g1v2.
    7 rows in crm_prd_info do not have matches in erp_px_cat_g1v2. They all fall into one category: CO-PE, which is not in erp_px_cat_g1v2. All of the products with this cate has the word "Pedal" in its name.
  Conclusion: erp_px_cat_g1v2 might be outdated.
*/

-- Find the number of matching categories.
SELECT
  COUNT(*)
FROM bronze.crm_prd_info c
INNER JOIN bronze.erp_px_cat_g1v2 e
ON REPLACE(LEFT(prd_key, 5), '-', '_') = e.id
;

-- Find the details of non-matching rows.
SELECT
  *
FROM bronze.crm_prd_info c
LEFT JOIN bronze.erp_px_cat_g1v2 e
  ON REPLACE(LEFT(c.prd_key, 5), '-', '_') = e.id
WHERE e.id IS NULL;

-- Find existing categories.
SELECT
DISTINCT cat
FROM bronze.erp_px_cat_g1v2
;

/*
  Data Validity
  Result: Success. All values have semantic meaning.
*/
-- Find all distinct values from cat
SELECT
  DISTINCT cat
FROM bronze.erp_px_cat_g1v2
;

-- Find all distinct values from subcat
SELECT
  DISTINCT subcat
FROM bronze.erp_px_cat_g1v2
;

-- Find all distinct values from maintenance
SELECT
  DISTINCT maintenance
FROM bronze.erp_px_cat_g1v2
;